In [4]:
import taichi as ti
ti.init(arch=ti.gpu)

import numpy as np

import time

[Taichi] Starting on arch=vulkan


Getting the data to compare to.

In [ ]:
data = np.genfromtxt("sim_data.csv", delimiter=",", skip_header=True)

data_texture = ti.field(float, data.shape)
data_texture.from_numpy(data.astype(np.float32))

array([[ 0.0000000e+00,  4.8723137e-03,  2.0125601e+00,  3.0723067e-03,
         1.2561004e-02],
       [ 3.3333335e-02,  1.4369836e-02,  1.8264962e+00,  1.9170964e-02,
        -1.7349805e-01],
       [ 6.6666670e-02,  2.8508434e-03,  1.9852126e+00,  1.4291546e-02,
        -1.4754671e-02],
       ...,
       [ 9.8999996e+00, -8.2529709e-02, -2.2164366e+00, -1.4824632e-01,
        -2.1751659e-01],
       [ 9.9333334e+00,  4.0721393e-01, -1.8323414e+00, -1.2686199e-02,
         1.2308279e-01],
       [ 9.9666662e+00,  4.8785990e-01, -1.7873330e+00, -2.7959490e-01,
         5.9559863e-02]], shape=(300, 5), dtype=float32)

Setting up the number of simulations happening at the same time, and for how long they should run.

In [6]:
num_sims = (800, 800)
sim_time = 50 #in seconds

In [7]:
sims = ti.Vector.field(2, ti.f32, shape=num_sims)
costs = ti.field(ti.f32, shape=num_sims)

Setting up initial conditions of the pendulums.

In [8]:
@ti.kernel
def setup():
    for i, j in sims:
        sims[i, j] = ti.Vector([ti.math.pi - 0.001, 0.1])
        costs[i, j] = 0

Helper Taichi functions

In [9]:
@ti.func
def map_range(value: float, min_in: float, max_in: float, min_out: float, max_out: float) -> float:
    return (value - min_in) / (max_in - min_in) * (max_out - min_out) + min_out

Defining the function $f$ in $y' = f(y)$

In [10]:
@ti.func
def get_angular_acceleration(y: ti.math.vec2, I: float, rcm: float, m: float, g: float) -> float : # type: ignore
    return -rcm*g*m*ti.sin(y[0])/I

@ti.func
def f(y: ti.math.vec2, I: float, rcm: float, m: float, g: float) -> ti.math.vec2:
    return ti.Vector([
        y[1],
        get_angular_acceleration(y, I, rcm, m, g)
    ])

A simulation step.

In [11]:
h = 0.005 # simulation time increment

@ti.kernel
def update():

    # setting parameters kept constant, ones we are pretty sure of
    m = 1; g = 9.81

    for i, j in sims:

        # setting parameters which change for each simulation
        I = map_range(i, 0, 500, 1, 2);
        rcm = map_range(j, 0, 500, 0.6, 1.4)

        oldValue = sims[i, j]
        k1 = h * f(oldValue, I, rcm, m, g)
        k2 = h * f(oldValue + k1/2, I, rcm, m, g)
        k3 = h * f(oldValue + k2/2, I, rcm, m, g)
        k4 = h * f(oldValue + k3, I, rcm, m, g)

        sims[i, j] = oldValue + k1/6 + k2/3 + k3/3 + k4/6

Determining the $\chi^2$ contribution for this data point. This kernel runs when a data point lies between this simulation step and the next (in time). A RK4 step is then performed to get the simulation to the time at which the data point was recorded. Finally, the position at the end of the pivot is calculated for the data point and the simulated point, and their squared distances are divided by the data uncertainty and added to the running $\chi^2$ value. This is all done for each simulation simultaneously.

Visualizing the result.

In [12]:
pixels = ti.Vector.field(3, ti.f32, shape=num_sims)

@ti.kernel
def draw():
    for i, j in pixels:
        pixels[i, j] = ti.Vector([sims[i, j][0]/10, sims[i, j][1]/10 + 0.5, 0])

Running the simulation loop.

In [13]:
window = ti.ui.Window("Visualization", num_sims)
canvas = window.get_canvas()

t = 0

setup()

for frame in range(int(sim_time/h)):
    if not window.running:
        break

    update()
    draw()

    t += h
    if (t % (1/60)) < ((t - h) % (1/60)):
        canvas.set_image(pixels)
        window.show()

window.destroy()